# Demo 1: Memory Stream System (ระบบความทรงจำ)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiraphap/Gen-Agent-Lectures-at-Econ-TU/blob/master/teaching/demos/demo_memory.ipynb)

## Overview

สคริปต์นี้สาธิตระบบ **Memory Stream** ที่เป็นหัวใจของ Generative Agents
อ้างอิงจากงานวิจัย Stanford UIST 2023

### หลักการสำคัญ
- Agent เก็บ "ความทรงจำ" ทุกอย่างที่เกิดขึ้น (observation, reflection, plan)
- เมื่อต้องตัดสินใจ จะ "ดึง" ความทรงจำที่เกี่ยวข้องที่สุดมาใช้
- การดึงใช้สูตร: **score = recency x importance x relevance**
- ไม่ต้องใช้ LLM ทั้งหมดเป็น pure Python

### ประเภทความทรงจำ
| ประเภท | ความหมาย | ตัวอย่าง |
|--------|----------|----------|
| **Observation** | สิ่งที่สังเกตเห็น | "เห็นโปรนมลดราคา" |
| **Reflection** | บทสะท้อน/สรุป | "Big C ราคาดีแต่จอดรถยาก" |
| **Plan** | แผนการ | "จะไปซื้อของวันเสาร์เช้า" |

> ไม่ต้องติดตั้งอะไรเพิ่ม ใช้แค่ standard library ของ Python

---
**ผศ.ดร.ถิรภาพ ฟักทอง** | คณะเศรษฐศาสตร์ มหาวิทยาลัยธรรมศาสตร์

In [ ]:
"""
Cell 2: Imports + Data Classes
ใช้ @dataclass เพื่อสร้าง class ที่เก็บข้อมูลได้สะดวก
Python จะสร้าง __init__, __repr__ ให้อัตโนมัติ
"""

import math
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, Optional


# ============================================================
# Data Classes สำหรับเก็บความทรงจำ
# ============================================================

@dataclass
class Memory:
    """โครงสร้างข้อมูลของความทรงจำ 1 ชิ้น

    แต่ละความทรงจำประกอบด้วย:
    - description: เนื้อหาของความทรงจำ (เช่น "เห็นโปรนมลด 30% ที่ Big C")
    - timestamp: เวลาที่เกิดเหตุการณ์
    - memory_type: ประเภท (observation=สิ่งที่เห็น, reflection=บทสะท้อน, plan=แผน)
    - importance: ความสำคัญ 1-10 (ปกติ LLM เป็นคนให้คะแนน แต่เราใส่เองใน demo นี้)
    - keywords: คำสำคัญสำหรับค้นหา (ปกติใช้ embedding แต่เราใช้ keyword overlap แทน)
    """
    description: str              # เนื้อหาความทรงจำ
    timestamp: datetime           # เวลาที่เกิดเหตุการณ์
    memory_type: str              # ประเภท: "observation", "reflection", "plan"
    importance: int               # ความสำคัญ 1-10
    keywords: List[str]           # คำสำคัญสำหรับค้นหา
    id: Optional[int] = None      # รหัสอ้างอิง (กำหนดอัตโนมัติ)


@dataclass
class ScoredMemory:
    """ความทรงจำที่ผ่านการให้คะแนนแล้ว ใช้ตอนแสดงผลการค้นหา

    เก็บทั้ง memory ต้นฉบับ และ score ย่อยแต่ละตัว
    เพื่อให้นักศึกษาเห็นว่าคะแนนมาจากไหน
    """
    memory: Memory
    recency_score: float      # คะแนนความใหม่ (0.0-1.0)
    importance_score: float   # คะแนนความสำคัญ (0.0-1.0)
    relevance_score: float    # คะแนนความเกี่ยวข้อง (0.0-1.0)
    total_score: float        # คะแนนรวม = recency × importance × relevance


print("Imports และ Data Classes พร้อมใช้งาน!")

## สูตร Retrieval (การดึงความทรงจำ)

เมื่อ agent ต้องตัดสินใจ จะ "ดึง" ความทรงจำที่เกี่ยวข้องที่สุดมาใช้ โดยใช้สูตร:

$$\text{score} = \text{recency} \times \text{importance} \times \text{relevance}$$

| องค์ประกอบ | สูตร | ความหมาย |
|-----------|------|----------|
| **Recency** (ความใหม่) | $0.99^{\text{hours}}$ | ยิ่งเพิ่งเกิด คะแนนยิ่งสูง (ลดลง 1% ทุกชั่วโมง) |
| **Importance** (ความสำคัญ) | $\text{score} / 10$ | LLM ให้คะแนน 1-10 ตามความสำคัญของเหตุการณ์ |
| **Relevance** (ความเกี่ยวข้อง) | $\text{keyword ตรง} / \text{keyword ทั้งหมด}$ | ในระบบจริงใช้ cosine similarity ของ embedding vectors |

### ทำไมต้องคูณกัน?
- คูณกัน = ต้องดีทุกด้าน ถ้าด้านใดด้านหนึ่งเป็น 0 ก็ตกไป
- ความทรงจำเก่ามากๆ ที่เกี่ยวข้อง -> recency ต่ำ ดึงได้น้อย
- ความทรงจำใหม่แต่ไม่เกี่ยวข้อง -> relevance ต่ำ ดึงได้น้อย
- ความทรงจำใหม่ + เกี่ยวข้อง + สำคัญ -> คะแนนสูงสุด!

In [ ]:
# ============================================================
# Recency (ความใหม่) - Exponential Decay
# ============================================================
# สูตร: recency = 0.99 ^ ชั่วโมงที่ผ่านไป
# ยิ่งเพิ่งเกิด คะแนนยิ่งสูง (ลดลง 1% ทุกชั่วโมง)
#
# ตัวอย่าง (decay_rate = 0.99):
#   - เพิ่งเกิด (0 ชม.): 0.99^0 = 1.000 (เต็ม!)
#   - 24 ชม. ก่อน (1 วัน): 0.99^24 = 0.785
#   - 168 ชม. ก่อน (1 สัปดาห์): 0.99^168 = 0.185
#
# → ความทรงจำเก่าจะถูก "ลืม" ไปเรื่อยๆ แต่ไม่เคยเป็น 0

decay_rate = 0.99

print("Recency Score ตามเวลาที่ผ่านไป:")
print("=" * 55)

test_hours = [0, 1, 6, 12, 24, 48, 72, 120, 168]
for h in test_hours:
    score = decay_rate ** h
    bar_len = int(score * 30)
    bar = "█" * bar_len + "░" * (30 - bar_len)
    label = ""
    if h == 0:
        label = "(เพิ่งเกิด)"
    elif h == 24:
        label = "(1 วัน)"
    elif h == 48:
        label = "(2 วัน)"
    elif h == 72:
        label = "(3 วัน)"
    elif h == 120:
        label = "(5 วัน)"
    elif h == 168:
        label = "(1 สัปดาห์)"
    print(f"  {h:>3} ชม. ก่อน: 0.99^{h:<3} = {score:.3f}  {bar}  {label}")

In [ ]:
# ============================================================
# Importance (ความสำคัญ)
# ============================================================
# สูตร: importance = raw_score / 10
#
# ในระบบจริง LLM จะให้คะแนนความสำคัญ 1-10:
#   1-3: เหตุการณ์ธรรมดาในชีวิตประจำวัน (เช่น "เดินผ่านร้าน")
#   4-6: เหตุการณ์ที่น่าสนใจ (เช่น "เห็นโปรลดราคา")
#   7-8: เหตุการณ์สำคัญ (เช่น "ซื้อของได้ราคาดีมาก")
#   9-10: เหตุการณ์ที่เปลี่ยนพฤติกรรม (เช่น "บริการแย่มาก ไม่ไปอีกแล้ว")
#
# ใน demo นี้เราใส่คะแนนเองเพื่อไม่ต้องเรียก LLM

print("Importance Score ตามคะแนนความสำคัญ:")
print("=" * 55)

examples = [
    (2, "เดินผ่านร้านสะดวกซื้อ"),
    (4, "เห็นป้ายโปรลดราคาหน้าร้าน"),
    (6, "เห็นใบปลิวโปรนมซื้อ 2 แถม 1"),
    (7, "ซื้อของที่ CJ More เห็นโปรนมลด 30%"),
    (8, "ลูกบ่นว่านมหมด ต้องซื้อด่วน"),
    (9, "โปรพิเศษนมสด ลด 30% เฉพาะสมาชิก"),
    (10, "บริการแย่มาก ไม่ไปร้านนี้อีกแล้ว"),
]

for score, desc in examples:
    norm = score / 10.0
    bar_len = int(norm * 30)
    bar = "█" * bar_len + "░" * (30 - bar_len)
    level = "ธรรมดา" if score <= 3 else "น่าสนใจ" if score <= 6 else "สำคัญ" if score <= 8 else "เปลี่ยนพฤติกรรม"
    print(f"  {score:>2}/10 = {norm:.1f}  {bar}  [{level}] {desc}")

In [ ]:
# ============================================================
# Relevance (ความเกี่ยวข้อง) - Keyword Overlap
# ============================================================
# สูตร: relevance = จำนวน keyword ที่ตรงกัน / จำนวน keyword ใน query
#
# ในระบบจริง ใช้ cosine similarity ของ embedding vectors:
#   - เอาข้อความผ่าน embedding model (เช่น text-embedding-ada-002)
#   - ได้ vector หลายมิติ
#   - คำนวณ cosine similarity ระหว่าง query กับ memory
#
# แต่ใน demo นี้ เราใช้ keyword overlap แทน เพื่อให้เข้าใจง่าย

print("Relevance Score - ตัวอย่างการคำนวณ:")
print("=" * 55)
print()

query_kw = ["นม", "โปร"]
print(f"  Query keywords: {query_kw}")
print()

test_memories = [
    (["นม", "ราคา", "โปร"], "เห็นโปรนมลด 30%"),
    (["นม", "หมด", "ลูก", "ซื้อ", "ด่วน"], "ลูกบ่นว่านมหมดแล้ว"),
    (["Big C", "จอดรถ", "เต็ม"], "ไป Big C ที่จอดรถเต็ม"),
    (["นม", "ไข่", "ซื้อ", "วันเสาร์", "โปร"], "วางแผนจะไปซื้อนมและไข่"),
]

for mem_kw, desc in test_memories:
    matched = [kw for kw in query_kw if kw.lower() in [k.lower() for k in mem_kw]]
    not_matched = [kw for kw in query_kw if kw.lower() not in [k.lower() for k in mem_kw]]
    score = len(matched) / len(query_kw)
    bar_len = int(score * 30)
    bar = "█" * bar_len + "░" * (30 - bar_len)

    print(f"  Memory: \"{desc}\"")
    print(f"    keywords: {mem_kw}")
    print(f"    ตรง: {matched}  ไม่ตรง: {not_matched}")
    print(f"    relevance = {len(matched)}/{len(query_kw)} = {score:.1f}  {bar}")
    print()

## MemoryStream Class - หัวใจของระบบ

`MemoryStream` คือ "สมอง" ของ Generative Agent เก็บทุกอย่างที่ agent ประสบมา และสามารถดึงความทรงจำที่เกี่ยวข้องที่สุดมาใช้ในการตัดสินใจ

### Method หลัก:
- `add_memory()` - เพิ่มความทรงจำใหม่
- `retrieve()` - ดึงความทรงจำที่เกี่ยวข้องที่สุด k อันดับแรก

### ค่า Decay Rate:
- `0.99` หมายความว่า ทุกๆ 1 ชั่วโมง คะแนน recency จะลดลง 1%
- ผ่านไป 24 ชม. -> 0.99^24 = 0.785 (เหลือ ~79%)
- ผ่านไป 168 ชม. (1 สัปดาห์) -> 0.99^168 = 0.185 (เหลือ ~19%)

In [ ]:
# ============================================================
# MemoryStream Class
# ============================================================

class MemoryStream:
    """ระบบจัดเก็บและค้นหาความทรงจำของ Agent

    Memory Stream คือ "สมอง" ของ Generative Agent
    เก็บทุกอย่างที่ agent ประสบมา และสามารถดึงความทรงจำ
    ที่เกี่ยวข้องที่สุดมาใช้ในการตัดสินใจ

    สูตรการให้คะแนน (Retrieval Formula):
        score = recency x importance x relevance
    """

    def __init__(self, owner_name: str):
        """สร้าง Memory Stream ใหม่

        Args:
            owner_name: ชื่อเจ้าของความทรงจำ (เช่น "สมศรี")
        """
        self.owner_name = owner_name
        self.memories: List[Memory] = []      # เก็บความทรงจำทั้งหมด
        self._next_id = 1                     # ตัวนับ ID อัตโนมัติ
        self.decay_rate = 0.99                # ค่า decay rate สำหรับคำนวณ recency

    def add_memory(self, description: str, timestamp: datetime,
                   memory_type: str, importance: int,
                   keywords: List[str]) -> Memory:
        """เพิ่มความทรงจำใหม่เข้าไปใน Memory Stream

        ทุกครั้งที่ agent ประสบเหตุการณ์ ไม่ว่าจะเป็น:
        - เห็นร้านค้า/โปรโมชั่น (observation)
        - คิดสรุปบทเรียน (reflection)
        - วางแผนจะทำอะไร (plan)
        จะถูกเก็บไว้ที่นี่
        """
        memory = Memory(
            description=description,
            timestamp=timestamp,
            memory_type=memory_type,
            importance=importance,
            keywords=keywords,
            id=self._next_id
        )
        self._next_id += 1
        self.memories.append(memory)
        return memory

    def _calculate_recency(self, memory: Memory, current_time: datetime) -> float:
        """คำนวณคะแนนความใหม่ (Recency Score)
        สูตร: recency = decay_rate ^ hours_since_event
        """
        time_diff = current_time - memory.timestamp
        hours_passed = time_diff.total_seconds() / 3600.0
        return self.decay_rate ** hours_passed

    def _calculate_importance(self, memory: Memory) -> float:
        """คำนวณคะแนนความสำคัญ (Importance Score)
        สูตร: importance = raw_score / 10
        """
        return memory.importance / 10.0

    def _calculate_relevance(self, memory: Memory, query_keywords: List[str]) -> float:
        """คำนวณคะแนนความเกี่ยวข้อง (Relevance Score)
        สูตร: relevance = จำนวน keyword ที่ตรงกัน / จำนวน keyword ใน query
        """
        if not query_keywords:
            return 0.0
        memory_kw_lower = [kw.lower() for kw in memory.keywords]
        query_kw_lower = [kw.lower() for kw in query_keywords]
        matches = sum(1 for qk in query_kw_lower if qk in memory_kw_lower)
        return matches / len(query_kw_lower)

    def retrieve(self, query_keywords: List[str], current_time: datetime,
                 top_k: int = 5) -> List[ScoredMemory]:
        """ดึงความทรงจำที่เกี่ยวข้องที่สุด k อันดับแรก

        นี่คือฟังก์ชันหลักของ Memory Stream!
        เมื่อ agent ต้องตัดสินใจ จะเรียกฟังก์ชันนี้
        เพื่อดึงความทรงจำที่ "เกี่ยวข้อง + สำคัญ + ใหม่" ที่สุด

        สูตรรวม: total_score = recency x importance x relevance
        """
        scored_memories = []

        for memory in self.memories:
            recency = self._calculate_recency(memory, current_time)
            importance = self._calculate_importance(memory)
            relevance = self._calculate_relevance(memory, query_keywords)
            total = recency * importance * relevance

            scored_memories.append(ScoredMemory(
                memory=memory,
                recency_score=recency,
                importance_score=importance,
                relevance_score=relevance,
                total_score=total
            ))

        scored_memories.sort(key=lambda x: x.total_score, reverse=True)
        return scored_memories[:top_k]


# ============================================================
# Helper functions สำหรับแสดงผล (ไม่ใช้สี ANSI)
# ============================================================

def make_bar(value: float, max_width: int = 30) -> str:
    """สร้าง bar chart ด้วย Unicode blocks"""
    filled = int(value * max_width)
    return "█" * filled + "░" * (max_width - filled)


def display_memory_addition(memory: Memory, index: int):
    """แสดงข้อมูลความทรงจำที่เพิ่มเข้าไป"""
    type_labels = {
        "observation": "สังเกต (Observation)",
        "reflection": "บทสะท้อน (Reflection)",
        "plan": "แผนการ (Plan)",
    }
    label = type_labels.get(memory.memory_type, memory.memory_type)
    print(f"  #{index:02d} [{label}] ความสำคัญ={memory.importance}/10  "
          f"({memory.timestamp.strftime('%d/%m %H:%M')})")
    print(f"      \"{memory.description}\"")
    print(f"      keywords: {memory.keywords}")


def display_retrieval_results(results: List[ScoredMemory], query: str,
                              query_keywords: List[str]):
    """แสดงผลลัพธ์การค้นหาแบบละเอียด พร้อม bar chart"""
    print(f"\n  Query: \"{query}\"")
    print(f"  Keywords ที่ใช้ค้นหา: {query_keywords}")
    print()

    if not results:
        print("  ไม่พบความทรงจำที่เกี่ยวข้อง")
        return

    for rank, sm in enumerate(results, 1):
        mem = sm.memory

        print(f"  อันดับ {rank} (score: {sm.total_score:.4f})")
        print(f"    \"{mem.description}\"")

        # Recency
        if sm.recency_score > 0:
            approx_hours = math.log(sm.recency_score) / math.log(0.99)
        else:
            approx_hours = float('inf')
        print(f"    ├─ Recency   (ความใหม่):      {sm.recency_score:.4f}  "
              f"(0.99^{approx_hours:.0f}ชม. = {sm.recency_score:.3f})")

        # Importance
        print(f"    ├─ Importance (ความสำคัญ):     {sm.importance_score:.4f}  "
              f"({mem.importance}/10 = {sm.importance_score:.1f})")

        # Relevance - แสดง keyword ที่ตรงกัน
        matched = [kw for kw in query_keywords if kw.lower() in [k.lower() for k in mem.keywords]]
        not_matched = [kw for kw in query_keywords if kw.lower() not in [k.lower() for k in mem.keywords]]
        match_str = ""
        if matched:
            match_str += f"ตรง: {matched} "
        if not_matched:
            match_str += f"ไม่ตรง: {not_matched}"
        print(f"    ├─ Relevance (ความเกี่ยวข้อง): {sm.relevance_score:.4f}  {match_str}")

        # Total
        print(f"    └─ Total = {sm.recency_score:.4f} x {sm.importance_score:.4f} x {sm.relevance_score:.4f} "
              f"= {sm.total_score:.4f}")

        # Bar chart
        print(f"       {make_bar(sm.total_score)} {sm.total_score:.4f}")
        print()


print("MemoryStream class และ helper functions พร้อมใช้งาน!")

In [ ]:
# ============================================================
# สร้างความทรงจำ 15 อันสำหรับ สมศรี (แม่บ้าน อายุ 45)
# ============================================================
#
# สมศรี มีลักษณะ:
# - price_sensitivity สูง (0.8) → สนใจเรื่องราคามาก
# - promotion_attraction สูง (0.9) → ชอบดูโปรโมชั่น
# - convenience_preference ต่ำ (0.4) → ไม่ถือเรื่องไกลมาก
# - impulse_buying ต่ำ (0.3) → ซื้อตามแผน ไม่ค่อยซื้อตามอารมณ์
#
# ความทรงจำจะครอบคลุมหลายช่วงเวลาและหลายประเภท
# เพื่อให้เห็นว่า recency, importance, relevance ส่งผลอย่างไร

# กำหนดเวลาฐาน (เสาร์เช้า 9:00 น.)
base_time = datetime(2024, 1, 13, 9, 0, 0)  # วันเสาร์ที่ 13 มกราคม 2024

print(f"เวลาปัจจุบันในการจำลอง: {base_time.strftime('%A %d/%m/%Y %H:%M')}")
print(f"(วันเสาร์เช้า - วันที่สมศรีชอบไปซื้อของ)")
print()

# สร้าง Memory Stream
stream = MemoryStream(owner_name="สมศรี")

# ---- ความทรงจำจาก 7 วันก่อน (168 ชม.) ----
# ความทรงจำเก่า: recency จะต่ำมาก (~0.185)
stream.add_memory(
    description="ไปซื้อของที่ CJ More เห็นโปรนม Dutch Mill ลด 30% ซื้อมา 3 กล่อง",
    timestamp=base_time - timedelta(days=7),
    memory_type="observation", importance=7,
    keywords=["นม", "โปร", "ลด", "CJ More", "Dutch Mill", "ซื้อ"])
stream.add_memory(
    description="พนักงาน CJ More บริการดีมาก ช่วยหิ้วของไปที่รถให้",
    timestamp=base_time - timedelta(days=7),
    memory_type="observation", importance=6,
    keywords=["CJ More", "บริการ", "พนักงาน", "ดี"])

# ---- ความทรงจำจาก 5 วันก่อน (120 ชม.) ----
# recency ~0.299
stream.add_memory(
    description="ลูกบ่นว่านมหมดแล้ว ต้องซื้อเพิ่มเร็วๆ",
    timestamp=base_time - timedelta(days=5),
    memory_type="observation", importance=8,
    keywords=["นม", "หมด", "ลูก", "ซื้อ", "ด่วน"])
stream.add_memory(
    description="เห็นใบปลิว Big C มีโปรลดนม Meiji ซื้อ 2 แถม 1",
    timestamp=base_time - timedelta(days=5),
    memory_type="observation", importance=7,
    keywords=["Big C", "โปร", "นม", "Meiji", "แถม", "ใบปลิว"])

# ---- ความทรงจำจาก 3 วันก่อน (72 ชม.) ----
# recency ~0.484
stream.add_memory(
    description="ไป Big C แต่ที่จอดรถเต็มมาก ต้องวนหาที่จอดนาน 20 นาที น่าหงุดหงิด",
    timestamp=base_time - timedelta(days=3),
    memory_type="observation", importance=6,
    keywords=["Big C", "จอดรถ", "เต็ม", "หงุดหงิด"])
stream.add_memory(
    description="สรุปว่า Big C ราคาดีแต่ที่จอดรถยากมาก โดยเฉพาะวันเสาร์",
    timestamp=base_time - timedelta(days=3),
    memory_type="reflection", importance=7,
    keywords=["Big C", "ราคา", "ดี", "จอดรถ", "ยาก", "วันเสาร์"])
stream.add_memory(
    description="ข้าวหมด ต้องซื้อข้าวหอมมะลิถุงใหญ่สำหรับครอบครัว",
    timestamp=base_time - timedelta(days=3),
    memory_type="observation", importance=8,
    keywords=["ข้าว", "หมด", "ซื้อ", "ครอบครัว", "หอมมะลิ"])

# ---- ความทรงจำจาก 2 วันก่อน (48 ชม.) ----
# recency ~0.617
stream.add_memory(
    description="แม่โทรมาบอกว่า Makro มีข้าวหอมมะลิราคาส่ง ถูกกว่าที่อื่นเยอะ",
    timestamp=base_time - timedelta(days=2),
    memory_type="observation", importance=6,
    keywords=["Makro", "ข้าว", "หอมมะลิ", "ราคา", "ถูก", "แม่"])
stream.add_memory(
    description="สรุปว่าของใช้ในครัวเหลือน้อย ต้องวางแผนไปซื้อของใหญ่สัปดาห์นี้",
    timestamp=base_time - timedelta(days=2),
    memory_type="plan", importance=7,
    keywords=["ซื้อ", "ครัว", "แผน", "สัปดาห์นี้", "ของใช้"])

# ---- ความทรงจำจาก 1 วันก่อน (24 ชม.) ----
# recency ~0.785
stream.add_memory(
    description="ดูใบปลิว CJ More เห็นโปรไข่ไก่ ซื้อ 2 แถม 1 และผงซักฟอกลด 25%",
    timestamp=base_time - timedelta(days=1),
    memory_type="observation", importance=7,
    keywords=["CJ More", "โปร", "ไข่", "แถม", "ผงซักฟอก", "ลด", "ใบปลิว"])
stream.add_memory(
    description="สามีบอกว่าผงซักฟอกใกล้หมดแล้ว ให้ช่วยซื้อด้วย",
    timestamp=base_time - timedelta(days=1),
    memory_type="observation", importance=5,
    keywords=["ผงซักฟอก", "หมด", "สามี", "ซื้อ"])

# ---- ความทรงจำจาก 6 ชั่วโมงก่อน ----
# recency ~0.941
stream.add_memory(
    description="เพื่อนบ้านบอกว่า 7-Eleven มีนม Dutch Mill ราคาปกติ ไม่มีโปร",
    timestamp=base_time - timedelta(hours=6),
    memory_type="observation", importance=3,
    keywords=["7-Eleven", "นม", "Dutch Mill", "ราคา", "ปกติ"])
stream.add_memory(
    description="วางแผนจะไปซื้อนมและไข่วันเสาร์เช้านี้ ต้องดูว่าร้านไหนโปรดีที่สุด",
    timestamp=base_time - timedelta(hours=6),
    memory_type="plan", importance=8,
    keywords=["นม", "ไข่", "ซื้อ", "วันเสาร์", "โปร", "แผน", "เช้า"])

# ---- ความทรงจำจาก 1 ชั่วโมงก่อน ----
# recency ~0.990
stream.add_memory(
    description="เปิดแอป CJ More เห็นว่าวันนี้มีโปรพิเศษนมสด ลด 30% เฉพาะสมาชิก",
    timestamp=base_time - timedelta(hours=1),
    memory_type="observation", importance=9,
    keywords=["CJ More", "โปร", "นม", "ลด", "สมาชิก", "พิเศษ", "แอป"])

# แสดงความทรงจำทั้งหมดที่เพิ่ม
print(f"กำลังเพิ่มความทรงจำ {len(stream.memories)} อัน...")
print("=" * 60)
for i, mem in enumerate(stream.memories, 1):
    display_memory_addition(mem, i)
print()
print(f"เพิ่มความทรงจำทั้งหมด {len(stream.memories)} อัน สำเร็จ!")

In [ ]:
# ============================================================
# Query 1: ค้นหาความทรงจำที่เกี่ยวกับ "นม โปร"
# ============================================================
# สถานการณ์: สมศรีกำลังคิดว่าจะซื้อนมที่ไหนดี
# คำถาม: มีความทรงจำอะไรเกี่ยวกับ "นม" และ "โปรโมชั่น" บ้าง?

query1 = "นม โปร"
query1_keywords = ["นม", "โปร"]

print("สถานการณ์: สมศรีกำลังคิดว่าจะซื้อนมที่ไหนดี")
print("คำถาม: มีความทรงจำอะไรเกี่ยวกับ \"นม\" และ \"โปรโมชั่น\" บ้าง?")
print()
print(f"กำลังคำนวณคะแนนความทรงจำทั้ง {len(stream.memories)} อัน...")

results1 = stream.retrieve(query1_keywords, base_time, top_k=5)

print(f"\nผลลัพธ์ Top-5:")
display_retrieval_results(results1, query1, query1_keywords)

# แสดงความทรงจำที่ไม่ติดอันดับ (score = 0) เพื่อให้เห็นว่าทำไม
all_results1 = stream.retrieve(query1_keywords, base_time, top_k=len(stream.memories))
zero_results = [r for r in all_results1 if r.total_score == 0]
if zero_results:
    print("ความทรงจำที่ได้คะแนน 0 (ไม่มี keyword ตรงกับ query เลย):")
    for sm in zero_results[:3]:
        print(f"  - \"{sm.memory.description[:50]}...\"")
        print(f"    keywords: {sm.memory.keywords} -> ไม่มีคำว่า {query1_keywords}")
    print()

In [ ]:
# ============================================================
# Query 2: ค้นหาความทรงจำที่เกี่ยวกับ "บริการ ดี ร้าน"
# ============================================================
# สถานการณ์: สมศรีกำลังเลือกร้าน อยากไปร้านที่บริการดี

query2 = "บริการดี ร้าน"
query2_keywords = ["บริการ", "ดี", "ร้าน"]

print("สถานการณ์: สมศรีกำลังเลือกร้าน อยากไปร้านที่บริการดี")
print("คำถาม: มีความทรงจำอะไรเกี่ยวกับ \"บริการดี\" และ \"ร้าน\" บ้าง?")

results2 = stream.retrieve(query2_keywords, base_time, top_k=5)

print(f"\nผลลัพธ์ Top-5:")
display_retrieval_results(results2, query2, query2_keywords)

In [ ]:
# ============================================================
# Query 3: ค้นหาความทรงจำที่เกี่ยวกับ "ข้าว ซื้อ ราคา"
# ============================================================
# สถานการณ์: สมศรีต้องซื้อข้าว อยากได้ราคาดี

query3 = "ข้าว ซื้อ ราคา"
query3_keywords = ["ข้าว", "ซื้อ", "ราคา"]

print("สถานการณ์: สมศรีต้องซื้อข้าว อยากได้ราคาดี")

results3 = stream.retrieve(query3_keywords, base_time, top_k=5)

print(f"\nผลลัพธ์ Top-5:")
display_retrieval_results(results3, query3, query3_keywords)

# ============================================================
# เปรียบเทียบว่า query ต่างกัน ผลลัพธ์ต่างกันอย่างไร
# ============================================================
print("=" * 60)
print("เปรียบเทียบผลลัพธ์จาก query ต่างกัน")
print("=" * 60)
print()

for query_name, results in [(query1, results1), (query2, results2), (query3, results3)]:
    print(f"  Query: \"{query_name}\"")
    for rank, sm in enumerate(results[:5], 1):
        short_desc = sm.memory.description[:45]
        if len(sm.memory.description) > 45:
            short_desc += "..."
        print(f"    {rank}. {make_bar(sm.total_score, 20)} "
              f"{sm.total_score:.4f} "
              f"{short_desc}")
    print()

## สรุป: สิ่งที่ได้เรียนรู้

### 1. Memory Stream
เก็บทุกสิ่งที่ agent ประสบมา แบ่งเป็น 3 ประเภท: **Observation**, **Reflection**, **Plan**

### 2. สูตร Retrieval
$$\text{score} = \text{recency} \times \text{importance} \times \text{relevance}$$

| องค์ประกอบ | ความหมาย | วิธีคำนวณ |
|-----------|----------|----------|
| **Recency** | ความทรงจำใหม่ = คะแนนสูง | Exponential decay ($0.99^{hours}$) |
| **Importance** | เหตุการณ์สำคัญ = คะแนนสูง | LLM ให้คะแนน 1-10 |
| **Relevance** | เกี่ยวข้องกับ query = คะแนนสูง | Embedding similarity |

### 3. Query ต่างกัน -> ผลลัพธ์ต่างกัน
- **"นม โปร"** -> ดึงความทรงจำเกี่ยวกับโปรนมมา
- **"บริการดี ร้าน"** -> ดึงความทรงจำเกี่ยวกับประสบการณ์ร้านค้า
- **"ข้าว ซื้อ ราคา"** -> ดึงความทรงจำเกี่ยวกับการซื้อข้าว

### 4. ในระบบจริง
- Relevance ใช้ **cosine similarity** ของ embedding vectors (ไม่ใช่ keyword overlap)
- Importance ใช้ **LLM ให้คะแนน** แต่ละเหตุการณ์ (ไม่ใช่กำหนดเอง)
- ใช้ **Vector Database** เช่น ChromaDB เก็บ embeddings

---

**ระบบนี้ทำให้ agent "จำ" ได้เหมือนคนจริง!**
ความทรงจำเก่าจะค่อยๆ จางไป แต่เหตุการณ์สำคัญจะยังคงถูกจำได้